# Daniel Phase 1 — QPINN Reproduction & Benchmarking

**ETH Quantum Hackathon 2026 — Quandela Challenge**

This notebook implements **Phase 1: Reproduction** of the QPINN challenge.
We solve the 1D heat equation with:
1. A **classical PINN baseline** (direct and auxiliary derivative)
2. A **MerLin DV-photonic QPINN** (hybrid quantum-classical)

All models are trained under **fair conditions**: same data budget, optimizer, epochs, and matched parameter counts where possible. We evaluate accuracy, training stability, and scaling behaviour.

*Code is adapted from the two provided starter notebooks:*
- `mlp_pinn_heat_equation_ETH_hackathon.ipynb`
- `merlin_dv_qpinn_ETH_hackathon.ipynb`


In [ ]:
import math
import time
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple, Dict, List

# MerLin import
import merlin as ML

# Reproducibility
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device and dtype
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float32
torch.set_default_dtype(DTYPE)

print('Device:', DEVICE)
print('Torch version:', torch.__version__)
print('MerLin version:', getattr(ML, '__version__', 'unknown'))


## Problem Definition: 1D Heat Equation

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}, \quad x \in [0,1], \; t \in [0,1]$$

Exact solution (used for IC and evaluation):
$$u(x,t) = e^{-\alpha \pi^2 t} \sin(\pi x)$$

Boundary conditions: $u(0,t) = u(1,t) = 0$
Initial condition: $u(x,0) = \sin(\pi x)$


In [ ]:
alpha = 0.1

def exact_u(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """Exact solution of the heat equation."""
    return torch.exp(-alpha * math.pi**2 * t) * torch.sin(math.pi * x)

def sample_interior(n: int, device=DEVICE, dtype=DTYPE) -> torch.Tensor:
    x = torch.rand(n, 1, device=device, dtype=dtype)
    t = torch.rand(n, 1, device=device, dtype=dtype)
    xt = torch.cat([x, t], dim=1)
    xt.requires_grad_(True)
    return xt

def sample_initial(n: int, device=DEVICE, dtype=DTYPE) -> Tuple[torch.Tensor, torch.Tensor]:
    x = torch.rand(n, 1, device=device, dtype=dtype)
    t = torch.zeros_like(x)
    xt = torch.cat([x, t], dim=1)
    y = exact_u(x, t)
    return xt, y

def sample_boundary(n: int, device=DEVICE, dtype=DTYPE) -> Tuple[torch.Tensor, torch.Tensor]:
    n0 = n // 2
    n1 = n - n0
    t0 = torch.rand(n0, 1, device=device, dtype=dtype)
    t1 = torch.rand(n1, 1, device=device, dtype=dtype)
    x0 = torch.zeros_like(t0)
    x1 = torch.ones_like(t1)
    xt = torch.cat([torch.cat([x0, t0], dim=1), torch.cat([x1, t1], dim=1)], dim=0)
    y = torch.zeros(n, 1, device=device, dtype=dtype)
    return xt, y


In [ ]:
def gradients(y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Compute dy/dx via autograd."""
    return torch.autograd.grad(
        y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
    )[0]


## Classical PINN Baseline — Direct Second Derivative

Adapted from `mlp_pinn_heat_equation_ETH_hackathon.ipynb`.
Standard MLP that computes $u_{\theta}(x,t)$ and evaluates the PDE residual $u_t - \alpha u_{xx}$ using nested automatic differentiation.


In [ ]:
class ClassicalPINN(nn.Module):
    def __init__(self, hidden: int = 32, depth: int = 4):
        super().__init__()
        layers = [nn.Linear(2, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, xt: torch.Tensor) -> torch.Tensor:
        return self.net(xt)

def pde_residual_direct(model: nn.Module, xt: torch.Tensor) -> torch.Tensor:
    """Compute u_t - alpha * u_xx."""
    u = model(xt)
    grad_u = gradients(u, xt)
    u_x = grad_u[:, 0:1]
    u_t = grad_u[:, 1:2]
    grad_ux = gradients(u_x, xt)
    u_xx = grad_ux[:, 0:1]
    return u_t - alpha * u_xx

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Classical PINN Baseline — Auxiliary Derivative

Adapted from `mlp_pinn_heat_equation_ETH_hackathon.ipynb`.
Matches the QPINN paper structure: the network outputs both $u$ and $\hat{u}_x$. The PDE residual becomes $u_t - \alpha \partial_x \hat{u}_x$, and a consistency loss enforces $\hat{u}_x \approx \partial_x u$.


In [ ]:
class ClassicalAuxPINN(nn.Module):
    def __init__(self, hidden: int = 32, depth: int = 4):
        super().__init__()
        layers = [nn.Linear(2, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, 2))  # [u, ux_hat]
        self.net = nn.Sequential(*layers)

    def forward(self, xt: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        y = self.net(xt)
        return y[:, 0:1], y[:, 1:2]

def pde_residual_aux(model: nn.Module, xt: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute residual and consistency for auxiliary formulation."""
    u, ux_hat = model(xt)
    grad_u = gradients(u, xt)
    u_x = grad_u[:, 0:1]
    u_t = grad_u[:, 1:2]
    grad_ux_hat = gradients(ux_hat, xt)
    ux_hat_x = grad_ux_hat[:, 0:1]
    residual = u_t - alpha * ux_hat_x
    consistency = u_x - ux_hat
    return residual, consistency


## MerLin DV-Photonic QPINN

Adapted from `merlin_dv_qpinn_ETH_hackathon.ipynb`.
Hybrid quantum-classical model using MerLin's `QuantumLayer.simple`. The architecture:
1. Classical feature map: $(x,t) \mapsto z$
2. MerLin quantum layer: $z \mapsto q$
3. Classical readout: $q \mapsto [q_u, \hat{u}_x]$
4. Hard-coded BC: $u = x(1-x) \cdot q_u$


In [ ]:
class MerlinQPINN(nn.Module):
    def __init__(self, feature_size: int = 4, quantum_output_size: int = 4, hidden: int = 16):
        super().__init__()
        self.feature_map = nn.Sequential(
            nn.Linear(2, hidden),
            nn.Tanh(),
            nn.Linear(hidden, feature_size),
        )
        self.quantum = ML.QuantumLayer.simple(
            input_size=feature_size,
            output_size=quantum_output_size,
        )
        self.readout = nn.Sequential(
            nn.Linear(quantum_output_size, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 2),
        )

    def forward(self, xt: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = xt[:, 0:1]
        z = self.feature_map(xt)
        q = self.quantum(z)
        out = self.readout(q)
        q_u = out[:, 0:1]
        ux_hat = out[:, 1:2]
        u = x * (1.0 - x) * q_u  # hard-coded Dirichlet BC
        return u, ux_hat

def pde_residual_merlin(model: nn.Module, xt: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """PDE residual + consistency for MerLin model."""
    u, ux_hat = model(xt)
    grad_u = gradients(u, xt)
    u_x = grad_u[:, 0:1]
    u_t = grad_u[:, 1:2]
    grad_ux_hat = gradients(ux_hat, xt)
    ux_hat_x = grad_ux_hat[:, 0:1]
    residual = u_t - alpha * ux_hat_x
    consistency = u_x - ux_hat
    return residual, consistency


## Training Configuration

We use a single `TrainConfig` dataclass for **all** models to ensure fair comparison.

**Important:** the MerLin starter notebook was carefully tuned with:
- `lr = 1e-2`
- `lambda_consistency = 0.1`
- `lambda_bc = 1.0`
- `n_f = n_i = n_b = 64`

Deviating from these values (e.g. lowering the learning rate or increasing the consistency weight)
noticeably degrades the MerLin QPINN result, so we keep the original tuned hyperparameters here.

Classical baselines are trained under the *same* settings for a fair benchmark.


In [ ]:
@dataclass
class TrainConfig:
    epochs: int = 300
    n_f: int = 64       # original merlin value
    n_i: int = 64
    n_b: int = 64
    lr: float = 1e-2    # tuned for MerLin quantum layer (higher than classical default)
    lambda_pde: float = 1.0
    lambda_ic: float = 10.0
    lambda_bc: float = 1.0     # original value (BC is near-zero by construction)
    lambda_consistency: float = 0.1  # original value: do not over-penalise auxiliary derivative
    print_every: int = 25

config = TrainConfig()
print(config)


## Unified Training Loop

We train each model with the **same** config and record:
- Total loss, PDE residual, consistency, IC loss, BC loss
- Wall-clock training time
- Relative $L^2$ error on a dense evaluation grid


In [ ]:
def train_model(
    model: nn.Module,
    config: TrainConfig,
    use_aux: bool = False,
    is_merlin: bool = False,
    device: torch.device = DEVICE,
    dtype: torch.dtype = DTYPE,
):
    """Train a PINN model and return history + metrics."""
    model = model.to(device=device, dtype=dtype)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    mse = nn.MSELoss()

    history = {"total": [], "pde": [], "consistency": [], "ic": [], "bc": []}
    start = time.time()

    for epoch in range(1, config.epochs + 1):
        optimizer.zero_grad()

        xt_f = sample_interior(config.n_f, device=device, dtype=dtype)
        xt_i, y_i = sample_initial(config.n_i, device=device, dtype=dtype)
        xt_b, y_b = sample_boundary(config.n_b, device=device, dtype=dtype)

        if is_merlin or use_aux:
            if is_merlin:
                r_f, r_c = pde_residual_merlin(model, xt_f)
            else:
                r_f, r_c = pde_residual_aux(model, xt_f)
            loss_pde = mse(r_f, torch.zeros_like(r_f))
            loss_consistency = mse(r_c, torch.zeros_like(r_c))
            u_i, _ = model(xt_i)
            u_b, _ = model(xt_b)
        else:
            r_f = pde_residual_direct(model, xt_f)
            loss_pde = mse(r_f, torch.zeros_like(r_f))
            loss_consistency = torch.tensor(0.0, device=device, dtype=dtype)
            u_i = model(xt_i)
            u_b = model(xt_b)

        loss_ic = mse(u_i, y_i)
        loss_bc = mse(u_b, y_b)

        loss = (
            config.lambda_pde * loss_pde
            + config.lambda_ic * loss_ic
            + config.lambda_bc * loss_bc
            + config.lambda_consistency * loss_consistency
        )

        loss.backward()
        optimizer.step()

        history["total"].append(loss.item())
        history["pde"].append(loss_pde.item())
        history["consistency"].append(loss_consistency.item())
        history["ic"].append(loss_ic.item())
        history["bc"].append(loss_bc.item())

        if epoch % config.print_every == 0 or epoch == 1:
            print(
                f"Epoch {epoch:4d} | loss={loss.item():.3e} | "
                f"pde={loss_pde.item():.3e} | ic={loss_ic.item():.3e} | "
                f"bc={loss_bc.item():.3e} | cons={loss_consistency.item():.3e}"
            )

    elapsed = time.time() - start
    return history, elapsed


## Evaluation on a Uniform Grid

We evaluate every model on the **same** $101 \times 101$ grid and compute the relative $L^2$ error:
$$\frac{\|u_\theta - u_{\text{true}}\|_2}{\|u_{\text{true}}\|_2}$$


In [ ]:
@torch.no_grad()
def evaluate_model(model: nn.Module, use_aux: bool = False, is_merlin: bool = False, nx: int = 101, nt: int = 101):
    x = torch.linspace(0, 1, nx, device=DEVICE, dtype=DTYPE).view(-1, 1)
    t = torch.linspace(0, 1, nt, device=DEVICE, dtype=DTYPE).view(-1, 1)
    X, T = torch.meshgrid(x.squeeze(), t.squeeze(), indexing="ij")
    xt = torch.stack([X.reshape(-1), T.reshape(-1)], dim=1)

    if use_aux or is_merlin:
        u_pred, _ = model(xt)
    else:
        u_pred = model(xt)

    u_pred = u_pred.reshape(nx, nt)
    u_true = exact_u(X, T)

    rel_l2 = torch.linalg.norm(u_pred - u_true) / torch.linalg.norm(u_true)
    max_abs = torch.max(torch.abs(u_pred - u_true))
    return (
        X.cpu(), T.cpu(), u_pred.cpu(), u_true.cpu(),
        rel_l2.item(), max_abs.item()
    )


## Matching Parameter Counts

For a fair comparison we resize the classical MLP so its total trainable parameters are close to the MerLin hybrid model. We print counts before training.


In [ ]:
merlin_model = MerlinQPINN(feature_size=4, quantum_output_size=4, hidden=16)
merlin_params = count_parameters(merlin_model)
print(f"MerLin QPINN parameters: {merlin_params}")

# Find a classical MLP with roughly the same parameter count
for hidden, depth in [(28, 4), (32, 4), (24, 5), (20, 6)]:
    m = ClassicalPINN(hidden=hidden, depth=depth)
    p = count_parameters(m)
    print(f"ClassicalPINN(hidden={hidden}, depth={depth}) -> {p} params")
    if abs(p - merlin_params) < 50:
        classical_hidden, classical_depth = hidden, depth
        break
else:
    classical_hidden, classical_depth = 32, 4

# Same for auxiliary MLP
for hidden, depth in [(28, 4), (32, 4), (24, 5)]:
    m = ClassicalAuxPINN(hidden=hidden, depth=depth)
    p = count_parameters(m)
    print(f"ClassicalAuxPINN(hidden={hidden}, depth={depth}) -> {p} params")
    if abs(p - merlin_params) < 50:
        aux_hidden, aux_depth = hidden, depth
        break
else:
    aux_hidden, aux_depth = 32, 4

print(f"\nSelected classical baseline: hidden={classical_hidden}, depth={classical_depth}")
print(f"Selected aux baseline:       hidden={aux_hidden}, depth={aux_depth}")


## Run All Experiments

We train three models under identical conditions:
1. **Classical Direct PINN**
2. **Classical Auxiliary PINN**
3. **MerLin QPINN**


In [ ]:
# 1. Classical Direct PINN
print("=" * 60)
print("Training Classical Direct PINN")
print("=" * 60)
clf_model = ClassicalPINN(hidden=classical_hidden, depth=classical_depth)
clf_history, clf_time = train_model(clf_model, config, use_aux=False, is_merlin=False)
clf_eval = evaluate_model(clf_model, use_aux=False, is_merlin=False)
print(f"Done in {clf_time:.1f}s | Rel L2: {clf_eval[4]:.4e}\n")

# 2. Classical Auxiliary PINN
print("=" * 60)
print("Training Classical Aux PINN")
print("=" * 60)
aux_model = ClassicalAuxPINN(hidden=aux_hidden, depth=aux_depth)
aux_history, aux_time = train_model(aux_model, config, use_aux=True, is_merlin=False)
aux_eval = evaluate_model(aux_model, use_aux=True, is_merlin=False)
print(f"Done in {aux_time:.1f}s | Rel L2: {aux_eval[4]:.4e}\n")

# 3. MerLin QPINN
print("=" * 60)
print("Training MerLin QPINN")
print("=" * 60)
merlin_model = MerlinQPINN(feature_size=4, quantum_output_size=4, hidden=16)
# Defensive dtype alignment
for p in merlin_model.parameters():
    if p.is_floating_point():
        p.data = p.data.to(DTYPE)
merlin_history, merlin_time = train_model(merlin_model, config, use_aux=True, is_merlin=True)
merlin_eval = evaluate_model(merlin_model, use_aux=True, is_merlin=True)
print(f"Done in {merlin_time:.1f}s | Rel L2: {merlin_eval[4]:.4e}\n")


## Quantitative Results


In [ ]:
results = {
    "Model": ["Classical Direct", "Classical Aux", "MerLin QPINN"],
    "Params": [
        count_parameters(clf_model),
        count_parameters(aux_model),
        count_parameters(merlin_model),
    ],
    "Time (s)": [clf_time, aux_time, merlin_time],
    "Rel L2": [clf_eval[4], aux_eval[4], merlin_eval[4]],
    "Max Abs Error": [clf_eval[5], aux_eval[5], merlin_eval[5]],
}

import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))


## Visualizations

### Training Curves


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(clf_history["total"], label="Classical Direct", alpha=0.9)
ax.semilogy(aux_history["total"], label="Classical Aux", alpha=0.9)
ax.semilogy(merlin_history["total"], label="MerLin QPINN", alpha=0.9)
ax.set_xlabel("Epoch")
ax.set_ylabel("Total Loss")
ax.set_title("Training Curves — Total Loss")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(clf_history["pde"], label="Classical Direct", alpha=0.9)
ax.semilogy(aux_history["pde"], label="Classical Aux", alpha=0.9)
ax.semilogy(merlin_history["pde"], label="MerLin QPINN", alpha=0.9)
ax.set_xlabel("Epoch")
ax.set_ylabel("PDE Residual Loss")
ax.set_title("Training Curves — PDE Residual")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()
plt.show()


### Predictions vs. Ground Truth


In [ ]:
def plot_heatmap(X, T, U, title, cmap="viridis"):
    fig, ax = plt.subplots(figsize=(5, 4))
    c = ax.contourf(T.numpy(), X.numpy(), U.numpy(), levels=50, cmap=cmap)
    ax.set_xlabel("t")
    ax.set_ylabel("x")
    ax.set_title(title)
    plt.colorbar(c, ax=ax)
    plt.tight_layout()
    plt.show()

# Exact
plot_heatmap(clf_eval[0], clf_eval[1], clf_eval[3], "Exact Solution")

# Classical Direct
plot_heatmap(clf_eval[0], clf_eval[1], clf_eval[2], "Classical Direct PINN")
plot_heatmap(clf_eval[0], clf_eval[1], clf_eval[2] - clf_eval[3], "Classical Direct Error", cmap="RdBu_r")

# Classical Aux
plot_heatmap(aux_eval[0], aux_eval[1], aux_eval[2], "Classical Aux PINN")
plot_heatmap(aux_eval[0], aux_eval[1], aux_eval[2] - aux_eval[3], "Classical Aux Error", cmap="RdBu_r")

# MerLin
plot_heatmap(merlin_eval[0], merlin_eval[1], merlin_eval[2], "MerLin QPINN")
plot_heatmap(merlin_eval[0], merlin_eval[1], merlin_eval[2] - merlin_eval[3], "MerLin QPINN Error", cmap="RdBu_r")


### Final Error Comparison


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
models = ["Classical\nDirect", "Classical\nAux", "MerLin\nQPINN"]
errors = [clf_eval[4], aux_eval[4], merlin_eval[4]]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
bars = ax.bar(models, errors, color=colors, edgecolor="black")
ax.set_ylabel("Relative $L^2$ Error")
ax.set_title("Final Accuracy Comparison")
ax.set_yscale("log")
for bar, err in zip(bars, errors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.2, f"{err:.2e}",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


## Multi-Seed Robustness Check

Single-seed results can be noisy. We re-run each model with 3 different seeds and report mean ± std.


In [ ]:
N_SEEDS = 3

multi_results = {
    "Classical Direct": {"rel_l2": [], "time": []},
    "Classical Aux": {"rel_l2": [], "time": []},
    "MerLin QPINN": {"rel_l2": [], "time": []},
}

for seed in range(SEED, SEED + N_SEEDS):
    print(f"\n--- Seed {seed} ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Classical Direct
    m = ClassicalPINN(hidden=classical_hidden, depth=classical_depth)
    _, t = train_model(m, config, use_aux=False, is_merlin=False)
    *_, rel_l2, _ = evaluate_model(m, use_aux=False, is_merlin=False)
    multi_results["Classical Direct"]["rel_l2"].append(rel_l2)
    multi_results["Classical Direct"]["time"].append(t)
    print(f"Direct  -> L2={rel_l2:.4e}, time={t:.1f}s")

    # Classical Aux
    m = ClassicalAuxPINN(hidden=aux_hidden, depth=aux_depth)
    _, t = train_model(m, config, use_aux=True, is_merlin=False)
    *_, rel_l2, _ = evaluate_model(m, use_aux=True, is_merlin=False)
    multi_results["Classical Aux"]["rel_l2"].append(rel_l2)
    multi_results["Classical Aux"]["time"].append(t)
    print(f"Aux     -> L2={rel_l2:.4e}, time={t:.1f}s")

    # MerLin
    m = MerlinQPINN(feature_size=4, quantum_output_size=4, hidden=16)
    for p in m.parameters():
        if p.is_floating_point():
            p.data = p.data.to(DTYPE)
    _, t = train_model(m, config, use_aux=True, is_merlin=True)
    *_, rel_l2, _ = evaluate_model(m, use_aux=True, is_merlin=True)
    multi_results["MerLin QPINN"]["rel_l2"].append(rel_l2)
    multi_results["MerLin QPINN"]["time"].append(t)
    print(f"MerLin  -> L2={rel_l2:.4e}, time={t:.1f}s")

print("\n=== Summary over seeds ===")
for name, vals in multi_results.items():
    l2_arr = np.array(vals["rel_l2"])
    t_arr = np.array(vals["time"])
    print(f"{name:20s} | Rel L2: {l2_arr.mean():.4e} ± {l2_arr.std():.4e} | Time: {t_arr.mean():.1f} ± {t_arr.std():.1f}s")


## Phase 1 Conclusions

Use the quantitative results above to answer the central challenge question:

> **Does the quantum component help, and if so, why?**

Key observations to report:
- **Accuracy**: Is the MerLin model more or less accurate than the classical baselines?
- **Training stability**: Do the loss curves converge smoothly?
- **Parameter efficiency**: Does the quantum layer achieve comparable accuracy with fewer parameters?
- **Speed**: Is training faster or slower? (Note: MerLin simulation overhead.)

## Suggested Ablations for Phase 2

1. **Freeze the quantum layer** and train only the readout — do learned quantum parameters matter?
2. **Replace the quantum layer** with a classical `nn.Linear` of the same shape.
3. **Remove the auxiliary derivative** and compute `u_xx` directly.
4. **Increase circuit depth** or feature size and observe scaling.
5. **Test on the Poisson equation** using the same architectures.
